In [1]:
import pandas as pd
import numpy as np
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
import tensorflow as tf

print("TensorFlow version:", tf.__version__)



print("GPUs detected:", tf.config.list_physical_devices('GPU'))

# Optional: check memory growth
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    for gpu in gpus:
        tf.config.experimental.set_memory_growth(gpu, True)



TensorFlow version: 2.20.0
GPUs detected: []


In [2]:
train_gen=ImageDataGenerator(rescale=1./255,shear_range=0.2,zoom_range=0.2,horizontal_flip=True)
trainning_setgen=train_gen.flow_from_directory(r'C:\Users\muham\Desktop\aiml\pneunomiadata\chest_xray\train',target_size=(150,150),batch_size=32,class_mode='binary', color_mode='grayscale')

test_gen=ImageDataGenerator(rescale=1./255)
test_setgen=test_gen.flow_from_directory(r'C:\Users\muham\Desktop\aiml\pneunomiadata\chest_xray\test',target_size=(150,150),batch_size=32,class_mode='binary',color_mode='grayscale',shuffle=False)

val_set = test_gen.flow_from_directory(
    r'C:\Users\muham\Desktop\aiml\pneunomiadata\chest_xray\val',
    target_size=(150,150),
    batch_size=32,
    class_mode='binary',
    color_mode='grayscale'
)


Found 5216 images belonging to 2 classes.
Found 624 images belonging to 2 classes.
Found 16 images belonging to 2 classes.


In [3]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPool2D, Flatten, Dense, Dropout
cnn = Sequential()

# 1st conv block
cnn = Sequential()

# 1st Block
cnn.add(Conv2D(32, (3,3), activation='relu', input_shape=(150,150,1)))
cnn.add(MaxPool2D(2,2))

# 2nd Block
cnn.add(Conv2D(64, (3,3), activation='relu'))
cnn.add(MaxPool2D(2,2))

# 3rd Block
cnn.add(Conv2D(128, (3,3), activation='relu'))
cnn.add(MaxPool2D(2,2))

# Flatten
cnn.add(Flatten())

# Dense Layers
cnn.add(Dense(256, activation='relu', name="feature_layer"))
cnn.add(Dropout(0.4))

cnn.add(Dense(1, activation='sigmoid'))


C:\Users\muham\anaconda3\Lib\site-packages\keras\src\layers\convolutional\base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [4]:
#cnn.add(tf.keras.layers.MaxPool2D(pool_size=2,strides=2))


In [5]:
#2nd convolution and pooloing layers

#cnn.add(tf.keras.layers.Conv2D(filters=32,kernel_size=3,activation='relu'))
#cnn.add(tf.keras.layers.MaxPool2D(pool_size=2,strides=2))

In [6]:
#cnn.add(tf.keras.layers.Flatten())
#cnn full conn
#cnn.add(tf.keras.layers.Dense(units=128,activation='relu'))
#cnn.add(tf.keras.layers.Dropout(0.5))

In [7]:
#cnn.add(tf.keras.layers.Dense(units=1,activation='sigmoid'))


In [8]:
#compiling
cnn.compile(optimizer='adam',loss='binary_crossentropy',metrics=['accuracy'])

In [9]:
#training
cnn.fit(x=trainning_setgen,validation_data=val_set,epochs=25)

C:\Users\muham\anaconda3\Lib\site-packages\keras\src\trainers\data_adapters\py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


Epoch 1/25
163/163 ━━━━━━━━━━━━━━━━━━━━ 147s 886ms/step - accuracy: 0.8351 - loss: 0.3733 - val_accuracy: 0.6875 - val_loss: 0.5721
Epoch 2/25
163/163 ━━━━━━━━━━━━━━━━━━━━ 201s 882ms/step - accuracy: 0.8961 - loss: 0.2563 - val_accuracy: 0.8750 - val_loss: 0.4057
Epoch 3/25
163/163 ━━━━━━━━━━━━━━━━━━━━ 203s 888ms/step - accuracy: 0.9091 - loss: 0.2260 - val_accuracy: 0.8125 - val_loss: 0.5117
Epoch 4/25
163/163 ━━━━━━━━━━━━━━━━━━━━ 200s 878ms/step - accuracy: 0.9254 - loss: 0.1901 - val_accuracy: 0.6250 - val_loss: 0.6382
Epoch 5/25
163/163 ━━━━━━━━━━━━━━━━━━━━ 144s 882ms/step - accuracy: 0.9350 - loss: 0.1687 - val_accuracy: 0.7500 - val_loss: 0.4936
Epoch 6/25
163/163 ━━━━━━━━━━━━━━━━━━━━ 253s 907ms/step - accuracy: 0.9425 - loss: 0.1572 - val_accuracy: 0.6250 - val_loss: 0.6527
Epoch 7/25
163/163 ━━━━━━━━━━━━━━━━━━━━ 201s 899ms/step - accuracy: 0.9450 - loss: 0.1497 - val_accuracy: 0.6250 - val_loss: 0.7971
Epoch 8/25
163/163 ━━━━━━━━━━━━━━━━━━━━ 208s 935ms/step - accuracy: 0.9477 -

In [ ]:
# Save the trained model

cnn.save("pneumonia_model1.h5")
print("✔ Model saved as pneumonia_model.h5")

# 3. Extract features for anomaly detection
from tensorflow.keras.models import Model
import numpy as np

print("Building model...")
_ = cnn.predict(np.zeros((1, 150, 150, 1)), verbose=0)

feature_model = Model(inputs=cnn.input, outputs=cnn.get_layer('feature_layer').output)

x_train_features = []
labels_list = []

print("Extracting features from training data...")
for i in range(len(trainning_setgen)):
    imgs, labels = trainning_setgen[i]
    feats = feature_model.predict(imgs, verbose=0)
    x_train_features.append(feats)
    labels_list.append(labels)
    
    if (i + 1) % 50 == 0:
        print(f"Processed {i + 1}/{len(trainning_setgen)} batches")

x_train_features = np.vstack(x_train_features)
labels_array = np.concatenate(labels_list)

print(f"Feature shape: {x_train_features.shape}")

# 4. Compute and save statistics
mean_vector = np.mean(x_train_features, axis=0)
cov_matrix = np.cov(x_train_features, rowvar=False)

from numpy.linalg import inv
inv_cov = inv(cov_matrix + 1e-5*np.eye(cov_matrix.shape[0]))

print("✔ Feature extraction complete!")
print(f"Mean vector shape: {mean_vector.shape}")
print(f"Inverse covariance shape: {inv_cov.shape}")

# 5. Save statistics for Streamlit
np.save('mean_vector.npy', mean_vector)
np.save('inv_cov.npy', inv_cov)
print("✔ Statistics saved to disk!")


In [ ]:
from scipy.spatial.distance import mahalanobis

def predict_xray_or_pneumonia(image):
    """
    Test function for predictions in notebook.
    image: preprocessed image of shape (150,150,1) scaled 0-1
    """
    # Extract features
    feat = feature_model.predict(np.expand_dims(image, axis=0), verbose=0)[0]
    
    # Mahalanobis distance
    dist = mahalanobis(feat, mean_vector, inv_cov)
    
    threshold = 10  # tune with validation X-rays
    if dist > threshold:
        return "Not an X-ray", 0, dist
    else:
        pred_prob = cnn.predict(np.expand_dims(image, axis=0), verbose=0)[0][0]
        label = "PNEUMONIA" if pred_prob > 0.5 else "NORMAL"
        return label, pred_prob, dist

# Test it on a sample image
# Example usage:
# from tensorflow.keras.preprocessing import image
# test_img = image.load_img('path/to/xray.jpg', target_size=(150,150), color_mode='grayscale')
# test_array = image.img_to_array(test_img) / 255.0
# result, confidence, distance = predict_xray_or_pneumonia(test_array)
# print(f"Prediction: {result}, Confidence: {confidence:.2%}, Distance: {distance:.2f}")

In [ ]:
import numpy as np
from sklearn.metrics import confusion_matrix, classification_report

# Predict probabilities
y_pred = cnn.predict(test_setgen)
# Convert probabilities to binary 0/1
y_pred_classes = (y_pred > 0.50).astype(int)
y_true = test_setgen.classes

# Confusion Matrix
cm = confusion_matrix(y_true, y_pred_classes)
print("Confusion Matrix:\n", cm)

# Classification Report
print("\nClassification Report:\n", classification_report(
    y_true, y_pred_classes, target_names=['NORMAL', 'PNEUMONIA']
))
